In [15]:
!pip install mumin

In [16]:
!pip install lxml_html_clean

In [31]:
import requests, zipfile, io, pandas as pd

# Télécharger le zip pré-compilé (claims inclus, pas de token nécessaire)
url = "https://data.bris.ac.uk/datasets/23yv276we2mll25fjakkfim2ml/mumin.zip"
print("Téléchargement en cours (~190 MB)...")

r = requests.get(url, stream=True)
z = zipfile.ZipFile(io.BytesIO(r.content))

# Voir le contenu du zip
print(z.namelist())

Téléchargement en cours (~190 MB)...
['tweet', 'claim', 'reply', 'user', 'article', 'user_follows_user', 'user_retweeted_tweet', 'reply_reply_to_tweet', 'reply_quote_of_tweet', 'tweet_discusses_claim', 'article_discusses_claim']


In [36]:
# Identifier le format réel du fichier
with open("claim", "rb") as f:
    header = f.read(16)
print(header)
print(header.hex())

b'\x80\x04\x95\x14\x01\x00\x00\x00\x00\x00\x00\x8c\x11pan'
80049514010000000000008c1170616e


In [3]:
import pickle
import pandas as pd

# Lire le fichier pickle
with open("claim", "rb") as f:
    claims_df = pickle.load(f)

print(type(claims_df))
print(claims_df.shape)
print(claims_df.columns.tolist())
print(claims_df.head())

<class 'pandas.core.frame.DataFrame'>
(12924, 18)
['id', 'keywords', 'cluster_keywords', 'cluster', 'date', 'language', 'embedding', 'label', 'reviewers', 'small_train_mask', 'small_val_mask', 'small_test_mask', 'medium_train_mask', 'medium_val_mask', 'medium_test_mask', 'large_train_mask', 'large_val_mask', 'large_test_mask']
       id                                  keywords  \
0  325195       corona virus reaching lungs remains   
1  325192                         big dubai airport   
2  325166           news corona virus vaccine ready   
3  325106  gardenzio stallone died morning prostate   
4  325096                81 blacks killed blacks 97   

                                    cluster_keywords cluster  \
0  coronavirus china covid 19 treatments recommended       0   
1                                                         -1   
2  coronavirus china covid 19 treatments recommended       0   
3                                                         -1   
4  agency jacques at

In [11]:
print(claims_df.shape)       # nombre de claims
print(claims_df.columns.tolist())  # colonnes
print(claims_df['label'].value_counts())  # misinformation vs factual

(11105, 18)
['id', 'keywords', 'cluster_keywords', 'cluster', 'date', 'language', 'embedding', 'label', 'reviewers', 'small_train_mask', 'small_val_mask', 'small_test_mask', 'medium_train_mask', 'medium_val_mask', 'medium_test_mask', 'large_train_mask', 'large_val_mask', 'large_test_mask']
label
misinformation    10657
factual             448
Name: count, dtype: int64


In [30]:
# Identifier le format réel du fichier
with open("claim_dump_large.pkl.xz", "rb") as f:
    header = f.read(16)
print(header)
print(header.hex())

b'\xfd7zXZ\x00\x00\x04\xe6\xd6\xb4F\x02\x00!\x01'
fd377a585a000004e6d6b44602002101


In [31]:
import pandas as pd

# Option 2 — pandas reads .pkl.xz directly
claims_df = pd.read_pickle("claim_dump_large.pkl.xz")

print(claims_df.shape)
print(claims_df.columns.tolist())
print(claims_df.head())

(13205, 6)
['claim', 'claim_en', 'verdict', 'train_mask', 'val_mask', 'test_mask']
                                               claim  \
0  วิดีโอประชาชนมาเลเซียแห่ซื้อของช่วงโควิด-19 ระ...   
1  Photo shows Myanmar military helicopter shot d...   
2  เฮลิคอปเตอร์กองทัพรัฐ​เมียนมาร์ถูกกบฎ KIA ยิงต...   
3  Video shows military helicopter shot down by r...   
4  'ฮ.พม่า'โดนสอยร่วงในรัฐคะฉิ่น นักบิน-ผู้โดยสาร...   

                                            claim_en         verdict  \
0  Video of Malaysian people flocking to buy thin...  misinformation   
1  Photo shows Myanmar military helicopter shot d...  misinformation   
2  A Myanmar State Army helicopter shot down by K...  misinformation   
3  Video shows military helicopter shot down by r...  misinformation   
4  'Myanmar helicopter' was taken down in Kachin ...  misinformation   

   train_mask  val_mask  test_mask  
0        True     False      False  
1       False     False       True  
2       False     False       True  

In [33]:
import pandas as pd

claims_df = pd.read_pickle("claim_dump_large.pkl.xz")
claims_df.to_csv("claims.csv", index=False)
print(f"Saved: {claims_df.shape[0]} rows × {claims_df.shape[1]} cols")

Saved: 13205 rows × 6 cols
